# DS 227 · Week 12 — Course synthesis
## Every course figure, reproduced in one run

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 13 — The run, reproduced: loading the data


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
text = requests.get(url, timeout=60).text      # kept: slide 3 reads it twice
df   = pd.read_csv(io.StringIO(text))
print(df.shape)

### Slide 14 — The run in code: weeks 4 to 7


Expected output:

```
w4  loaded                 34079
w4  amount_paid unique     [0]
w5  naive dropna()         25370 (25.6% lost)
w5  completed, true        80.8%
w5  completed, after drop  99.6%
w6  impossible durations   1
w7  describe() count       27664
w7  budget skew            5.13
```


In [ ]:
# weeks 4-7, one line each
print("w4  loaded                ", len(df))
print("w4  amount_paid unique    ", df.amount_paid.unique())
d = df.dropna()
print("w5  naive dropna()        ", len(d), f"({1 - len(d) / len(df):.1%} lost)")
print("w5  completed, true       ", f"{(df.status == 'Completed').mean():.1%}")
print("w5  completed, after drop ", f"{(d.status == 'Completed').mean():.1%}")

for col in ["start_date", "completion_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["duration"] = (df.completion_date - df.start_date).dt.days
print("w6  impossible durations  ", (df.duration < 0).sum())
print("w7  describe() count      ", int(df.duration.describe()["count"]))
print("w7  budget skew           ", round(df.budget.skew(), 2))

### Slide 15 — The run in code: weeks 8 to 11


Expected output:

```
w8  pearson / spearman     0.365 / 0.508
w8  median days by quartile [119, 189, 226, 269]
w9  contractors            4841
w9  top 1% share           27.6%
w11 unique at 1M           30.9%
w11 50M bands + region     0.4%
```


In [ ]:
# weeks 8-11, continuing from the previous slide
sub = df[["budget", "duration"]].dropna()
r = sub.budget.corr(sub.duration)
rho = sub.budget.rank().corr(sub.duration.rank())
print("w8  pearson / spearman    ", round(r, 3), "/", round(rho, 3))
q = pd.qcut(sub.budget, 4, labels=False)
print("w8  median days by quartile", sub.groupby(q).duration.median().astype(int).tolist())

c = df.groupby("contractor").budget.sum().sort_values(ascending=False)
print("w9  contractors           ", len(c))
print("w9  top 1% share          ", f"{c.head(48).sum() / c.sum():.1%}")

def alone(cols):
    g = df.groupby(cols).size()
    return f"{(g == 1).sum() / len(df):.1%}"
df["b"] = (df.budget / 1e6).round()
print("w11 unique at 1M          ", alone(["region", "province", "year", "b"]))
df["b"] = (df.budget / 5e7).round() * 50
print("w11 50M bands + region    ", alone(["region", "year", "b"]))

### Slide 16 — The error was in the synthesis script


Expected output:

```
27664
25370
```


In [ ]:
import csv

# DictReader: every cell is text, and an empty cell is "" (a value)
rows = pd.DataFrame(list(csv.DictReader(io.StringIO(text))))
for col in ["start_date", "completion_date"]:
    rows[col] = pd.to_datetime(rows[col], errors="coerce")   # "" becomes NaT
print(len(rows.dropna()))

# read_csv: an empty cell becomes NaN, which dropna() removes
print(len(pd.read_csv(io.StringIO(text)).dropna()))

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
